In [ ]:
import json
import pandas as pd
import numpy as np

In [50]:
with open("dataset_filter.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for item in data:
    rows.append({
        "username": item["authorMeta"]["name"],
        "followers": item["authorMeta"]["fans"],

        "create_time": pd.to_datetime(item["createTime"], unit="s"),
        "duration": item["videoMeta"]["duration"],

        "views": item["playCount"],
        "likes": item["diggCount"],
        "comments": item["commentCount"],
        "shares": item["shareCount"],

        "caption": item["text"],
        "hashtag": item["hashtags"],
        "musicOriginal": item['musicMeta']['musicOriginal'],
        "coverUrl": item['videoMeta']['coverUrl']
    })

df = pd.DataFrame(rows)
df = df.sort_values(by=["username", "create_time"], ascending=[True, True])

data = df

In [ ]:
import pandas as pd
import numpy as np

def format_num(num):
    try:
        num = float(num)
    except (ValueError, TypeError):
        return "0"
    if num >= 1_000_000: return f"{num/1_000_000:.1f}M"
    if num >= 1_000: return f"{num/1_000:.1f}k"
    return str(int(num))

def build_bert_sequences(df, window_size=10):
    sequences = []
    weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    df = df.copy()
    df['hashtag_str'] = df['hashtag'].apply(
        lambda x: " ".join([tag['name'] for tag in x]) if isinstance(x, list) else ""
    )
    df['caption'] = df['caption'].fillna("").str.replace("\n", " ").str.strip()
    
    for username, group in df.groupby("username"):
        group = group.sort_values("create_time")
        if len(group) <= window_size:
            continue
            
        for i in range(len(group) - window_size):
            history = group.iloc[i : i + window_size]
            target = group.iloc[i + window_size]
            
            f_count = format_num(target['followers'])
            t_day = weekdays[target['create_time'].weekday()]
            t_hour = target['create_time'].hour
            t_tags = "#" + target['hashtag_str'].replace(' ', ' #') if target['hashtag_str'] else "#"
            
            main_header = (f"Account: {f_count} followers. "
                           f"Target Video: {t_day} {t_hour}h, {target['duration']}s, "
                           f"tags: {t_tags}, caption: \"{target['caption'][:100]}\".")
            
            history_segments = []
            for idx, (_, h_row) in enumerate(history.iterrows()):
                h_day = weekdays[h_row['create_time'].weekday()]
                h_hour = h_row['create_time'].hour
                h_v = format_num(h_row['views'])
                h_l = format_num(h_row['likes'])
                h_c = format_num(h_row['comments'])
                h_s = format_num(h_row['shares'])
                h_tags = "#" + h_row['hashtag_str'].replace(' ', ' #') if h_row['hashtag_str'] else "#"
                
                h_str = (f"History {idx+1}: {h_day} {h_hour}h, {h_v} views, {h_row['duration']}s, "
                         f"{h_l} likes, {h_c} comments, {h_s} shares, "
                         f"tags: {h_tags}, caption: \"{h_row['caption'][:40]}\".")
                history_segments.append(h_str)
            
            full_text = main_header + " " + " ".join(history_segments)
            
            explosion_score = target['views'] / (target['followers'])
            log_target = np.log1p(explosion_score)
            
            sequences.append({
                "username": username,
                "text": full_text,
                "label": float(log_target),
                "raw_views": target['views']
            })
            
    return pd.DataFrame(sequences)

bert_df = build_bert_sequences(df)

print(bert_df.iloc[1])

username                                          .sturdy.manz
text         Account: 4.5k followers. Target Video: Tue 3h,...
label                                                 0.820464
raw_views                                                 5708
Name: 1, dtype: object


In [49]:
print(bert_df.iloc[1]['text'])

Account: 4.5k followers. Target Video: Tue 3h, 15s, tags: #fypシ゚, caption: "how df i get here? #fypシ゚". History 1: Fri 0h, 67.3k views, 0s, 6.8k likes, 0 comments, 683 shares, tags: #, caption: "even tho we ain’t blood". History 2: Sat 16h, 3.3k views, 9s, 169 likes, 0 comments, 2 shares, tags: #fyp, caption: "👤 #fyp". History 3: Sun 18h, 3.7k views, 15s, 181 likes, 1 comments, 2 shares, tags: #fyp, caption: "Pt.10 this dance fun asf #fyp". History 4: Sun 18h, 3.4k views, 15s, 141 likes, 5 comments, 1 shares, tags: #fyp, caption: "everyday tbh #fyp". History 5: Tue 5h, 4.3k views, 24s, 279 likes, 0 comments, 6 shares, tags: #fyp, caption: "gonna be a long few years, sometimes i w". History 6: Fri 2h, 6.0k views, 15s, 331 likes, 9 comments, 4 shares, tags: #fyp, caption: "can’t stop doing this dance #fyp". History 7: Tue 20h, 16.8k views, 0s, 1.2k likes, 11 comments, 12 shares, tags: #, caption: "slow motion better then no motion😤". History 8: Mon 20h, 5.3k views, 15s, 233 likes, 10 com

In [ ]:
Account: 4.5k followers. Target Video: Tue 3h, 15s, tags: #fypシ゚, caption: "how df i get here? #fypシ゚". History 1: Fri 0h, 67.3k views, 0s, 6.8k likes, 0 comments, 683 shares, tags: #, caption: "even tho we ain’t blood". History 2: Sat 16h, 3.3k views, 9s, 169 likes, 0 comments, 2 shares, tags: #fyp, caption: "👤 #fyp". History 3: Sun 18h, 3.7k views, 15s, 181 likes, 1 comments, 2 shares, tags: #fyp, caption: "Pt.10 this dance fun asf #fyp". History 4: Sun 18h, 3.4k views, 15s, 141 likes, 5 comments, 1 shares, tags: #fyp, caption: "everyday tbh #fyp". History 5: Tue 5h, 4.3k views, 24s, 279 likes, 0 comments, 6 shares, tags: #fyp, caption: "gonna be a long few years, sometimes i w". History 6: Fri 2h, 6.0k views, 15s, 331 likes, 9 comments, 4 shares, tags: #fyp, caption: "can’t stop doing this dance #fyp". History 7: Tue 20h, 16.8k views, 0s, 1.2k likes, 11 comments, 12 shares, tags: #, caption: "slow motion better then no motion😤". History 8: Mon 20h, 5.3k views, 15s, 233 likes, 10 comments, 6 shares, tags: #fyp, caption: "yes in 25 idc.😐 #fyp". History 9: Wed 13h, 13.9k views, 0s, 1.3k likes, 20 comments, 19 shares, tags: #draft #fypシ゚, caption: "#draft #fypシ゚  mb for late ahh post, bee". History 10: Tue 23h, 10.5k views, 21s, 869 likes, 10 comments, 22 shares, tags: #fypシ゚ #fyp, caption: "my camera shaking🥲 #fypシ゚ #fyp".